# Parte 4 — Experimento B: BERTweet
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `vinai/bertweet-base`  
**Pre-entrenamiento:** 850M tweets en inglés  
**Tokenizador:** BPE entrenado sobre texto de Twitter

BERTweet tiene la misma arquitectura que BERT-base (12 capas, 768 dimensiones, ~110M parámetros), pero fue pre-entrenado desde cero sobre tweets.

> **Nota sobre `normalization=True`:** el tokenizador normaliza las URLs a `HTTPURL` y los @mentions a `@USER` antes de tokenizar, replicando el preprocesamiento usado durante el pre-entrenamiento.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
%run part-2-pipeline.ipynb

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 4.1 — Tokenizar el dataset con BERTweet

Importante: carga el tokenizador con `normalization=True`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)


### 📝 TODO 4.2 — Cargar el modelo BERTweet

Mismo procedimiento que con DistilBERT pero con `MODEL_CHECKPOINT`. Imprime el número de parámetros y compáralo con DistilBERT.

In [ ]:
model_bertweet = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
total = sum(p.numel() for p in model_bertweet.parameters())
trainable = sum(p.numel() for p in model_bertweet.parameters() if p.requires_grad)
print(f"Parámetros totales: {total/1e6:.1f}M | entrenables: {trainable/1e6:.1f}M")


### 📝 TODO 4.3 — Entrenar y evaluar BERTweet

Repite el mismo proceso del TODO 3.3 con el modelo y dataset de BERTweet.

**Pregunta:** ¿necesitarías cambiar el learning rate para BERTweet vs DistilBERT? ¿Por qué sí o por qué no?

### Respuesta — ¿Cambiar el learning rate respecto a DistilBERT?

**En general, no hace falta cambiarlo** para el fine-tuning completo: usamos el mismo `LR = 2e-5` en ambos experimentos.

**Por qué:**  
- Ambos son transformers del estorno Hugging Face con **capas de clasificación nuevas** y el resto pre-entrenado.  
- `2e-5` es el valor estándar de BERT/BERTweet para fine-tuning en tareas de clasificación.  
- BERTweet tiene más parámetros (~110M vs ~66M de DistilBERT), pero el LR bajo evita destruir representaciones ya útiles para Twitter.

**Cuándo sí cambiaría el LR:**  
- **LoRA** (parte 5): subimos a `2e-4` porque solo entrenamos ~0.65% de parámetros (adaptadores), no todo el modelo.  
- Si el modelo **no converge** o diverge: bajar a `1e-5`.  
- Si **underfitting** con muchas épocas: probar `3e-5` con early stopping.

**Resultados obtenidos (1 época, CPU/Kaggle):** DistilBERT test acc. **69.5%**; BERTweet **76.6%** — sin ajustar LR, BERTweet gana por mejor alineación dominio/tokenización, no por el valor del learning rate.

In [ ]:
trainer_bertweet = make_trainer(
    model_bertweet, tok_bertweet, ds_bertweet,
    output_dir="./checkpoints/bertweet", lr=LR,
)
trainer_bertweet.train()
plot_training_curves(trainer_bertweet, title="BERTweet")
metrics_bertweet = full_evaluation(trainer_bertweet, ds_bertweet["test"], model_name="BERTweet")
metrics_bertweet


### Resultados e interpretación — BERTweet

| Métrica (test) | Valor aprox. |
|----------------|--------------|
| Accuracy | **76.6%** |
| F1 macro (validación) | **~0.64** |

**Interpretación:** BERTweet supera a DistilBERT (~+7 puntos de accuracy) porque el pre-entrenamiento en **850M tweets** y la normalización `@USER` / hashtags mejoran la representación del input. Es el **mejor modelo** de los tres en este taller con fine-tuning completo.

## Push to Hub

In [ ]:
import os
if os.environ.get("HF_TOKEN"):
    trainer_bertweet.push_to_hub(HF_REPO, commit_message="BERTweet — TweetEval emotion")
    print(f"Modelo en https://huggingface.co/{HF_REPO}")
else:
    trainer_bertweet.save_model("./checkpoints/bertweet/best")
    print("Guardado localmente (sin HF_TOKEN).")
